# Preparación de parcelas reales

Este cuaderno organiza el catálogo, guarda las geometrías y realiza controles previos a la extracción satelital.

## Cómo utilizarlo
Ejecutar las celdas en orden en Google Colab. Los datos privados de las parcelas (coordenadas, municipio y variedad) no están en este notebook: se leen de un archivo `datos_parcelas_privados.py` que se guarda en la carpeta `privado` de Drive y no se sube al repositorio. En el repositorio hay una plantilla con valores ficticios, `datos_parcelas_privados_ejemplo.py`, que se copia con ese nombre y se rellena con los datos reales.

## Entradas y salidas
Entradas: el archivo `datos_parcelas_privados.py` con los nombres, municipios, variedades y polígonos de las parcelas. Salidas: cuatro GeoJSON, catálogo CSV, informes de revisión y registro privado del trabajo, todos en Drive. El siguiente notebook utiliza estos archivos desde Drive.

## Límites
No incorpora históricos, meteorología ni decisiones de riego. La validez geométrica no certifica la campaña ni el cultivo. Los archivos del mismo nombre pueden actualizarse al repetir la ejecución. Las salidas de este notebook muestran datos de las explotaciones: no deben publicarse en el repositorio.


## 1. Entorno de trabajo y almacenamiento

Importa las librerías, monta Google Drive, define las carpetas del proyecto y carga los datos privados de las parcelas desde `privado/datos_parcelas_privados.py`. Autoriza el acceso cuando Colab lo solicite. Las carpetas se crean si no existen. Si falta el archivo privado, el notebook se detiene con un aviso. Los archivos se guardan en Drive, no en GitHub.


In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

base_path = Path(
    "/content/drive/MyDrive/TFG_PATATAS_TRABAJO"
)

parcelas_path = base_path / "parcelas"
resultados_path = base_path / "resultados"
privado_path = base_path / "privado"

parcelas_path.mkdir(parents=True, exist_ok=True)
resultados_path.mkdir(parents=True, exist_ok=True)

if not (privado_path / "datos_parcelas_privados.py").exists():
    raise FileNotFoundError(
        "Falta datos_parcelas_privados.py en la carpeta privado de Drive. "
        "Copia datos_parcelas_privados_ejemplo.py con ese nombre y rellénalo "
        "con los datos reales de las parcelas. No se sube al repositorio."
    )

sys.path.insert(0, str(privado_path))

from datos_parcelas_privados import CATALOGO, GEOMETRIAS


## 2. Catálogo operativo de parcelas

Construye el catálogo de Pancostales, Puri, La Culebra y Los Ceres a partir de `CATALOGO`, con identificadores estables, municipio, variedad y nombre de GeoJSON. Los Ceres-1 y Los Ceres-2 se han unificado según la confirmación del estudiante. La campaña, siembra y emergencia siguen sin confirmar; no deben rellenarse por inferencia a partir de las imágenes. La tabla que se muestra contiene municipio y variedad: no publicar este notebook con salidas.


In [ ]:
catalogo = pd.DataFrame(CATALOGO)

catalogo["campania"] = pd.NA
catalogo["fecha_siembra"] = pd.NaT
catalogo["fecha_emergencia"] = pd.NaT
catalogo["estado_revision"] = "Pendiente"

display(catalogo)


## 3. Función de guardado de geometrías

Convierte el texto JSON en un objeto, comprueba que sea Polygon y revisa el anillo exterior, su cierre y los rangos de longitud y latitud. Escribe un archivo por parcela. Esta validación básica no sustituye a la comprobación topológica posterior con Shapely. Repetir el guardado actualiza el archivo del mismo nombre.


In [ ]:
def guardar_geometria(parcela_id, texto_geojson):
    geometria = json.loads(texto_geojson)

    if geometria.get("type") != "Polygon":
        raise ValueError("Se esperaba una geometría de tipo Polygon.")

    coordenadas = geometria.get("coordinates")

    if not coordenadas or not coordenadas[0]:
        raise ValueError("El polígono no contiene coordenadas.")

    anillo = coordenadas[0]

    if len(anillo) < 4:
        raise ValueError("El anillo necesita al menos cuatro posiciones.")

    if anillo[0] != anillo[-1]:
        raise ValueError("El anillo no está cerrado.")

    for posicion in anillo:
        if len(posicion) < 2:
            raise ValueError("Una posición no tiene longitud y latitud.")

        longitud, latitud = posicion[:2]

        if not (-180 <= longitud <= 180):
            raise ValueError("Longitud fuera de rango.")

        if not (-90 <= latitud <= 90):
            raise ValueError("Latitud fuera de rango.")

    archivo = parcelas_path / f"{parcela_id}.geojson"

    archivo.write_text(
        json.dumps(geometria, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    return archivo

## 4. Geometría de Pancostales

Toma la geometría de Pancostales de `GEOMETRIAS` (archivo privado) y crea pancostales.geojson en Drive. El orden de cada posición es longitud, latitud. Ejecutar después de definir guardar_geometria y las rutas.


In [ ]:
texto_pancostales = GEOMETRIAS["pancostales"]

archivo = guardar_geometria(
    "pancostales",
    texto_pancostales
)

print("Geometría guardada:", archivo.name)

## 5. Geometría de Puri

Toma la geometría de Puri de `GEOMETRIAS` (archivo privado) y crea puri.geojson en Drive. El orden de cada posición es longitud, latitud. Ejecutar después de definir guardar_geometria y las rutas.


In [ ]:
texto_puri = GEOMETRIAS["puri"]

archivo = guardar_geometria(
    "puri",
    texto_puri
)

print("Geometría guardada:", archivo.name)

## 6. Geometría de La Culebra

Toma la geometría de La Culebra de `GEOMETRIAS` (archivo privado) y crea la_culebra.geojson en Drive. El orden de cada posición es longitud, latitud. Ejecutar después de definir guardar_geometria y las rutas.


In [ ]:
texto_la_culebra = GEOMETRIAS["la_culebra"]

archivo = guardar_geometria(
    "la_culebra",
    texto_la_culebra
)

print("Geometría guardada:", archivo.name)

## 7. Geometría unificada de Los Ceres

Toma la geometría de Los Ceres de `GEOMETRIAS` (archivo privado) y crea los_ceres.geojson en Drive. El orden de cada posición es longitud, latitud. Ejecutar después de definir guardar_geometria y las rutas.


In [ ]:
texto_los_ceres = GEOMETRIAS["los_ceres"]

archivo = guardar_geometria(
    "los_ceres",
    texto_los_ceres
)

print("Geometría guardada:", archivo.name)

## 8. Disponibilidad de archivos

Comprueba que existan los cuatro GeoJSON esperados. Si falta alguno, detiene la ejecución. La existencia de un archivo no demuestra que su geometría sea válida; eso se comprueba después.


In [ ]:
archivos_esperados = [
    "pancostales.geojson",
    "puri.geojson",
    "la_culebra.geojson",
    "los_ceres.geojson"
]

faltantes = []

for nombre in archivos_esperados:
    archivo = parcelas_path / nombre

    if archivo.exists():
        print(nombre, "→ Guardado")
    else:
        print(nombre, "→ FALTA")
        faltantes.append(nombre)

if faltantes:
    raise FileNotFoundError(
        "Faltan archivos: " + ", ".join(faltantes)
    )

print("\nLos cuatro archivos están disponibles.")

## 9. Dependencia para validación geométrica

Instala Shapely en el entorno de Colab. Esta celda prepara la librería utilizada en las siguientes comprobaciones; no modifica las coordenadas.


In [ ]:
%pip -q install shapely

## 10. Revisión geométrica del catálogo

Lee cada geometría, comprueba su tipo, validez topológica y que no esté vacía. Genera revision_df y carga las geometrías válidas. Si faltan archivos o hay geometrías no válidas, se detiene. «Geometría validada» no significa que se hayan validado municipio, cultivo, campaña o límites catastrales.


In [ ]:
from shapely.geometry import shape
from shapely.validation import explain_validity

geometrias = {}
revision = []

for fila in catalogo.to_dict("records"):
    archivo = parcelas_path / fila["archivo_geometria"]

    resultado = {
        "parcela_id": fila["parcela_id"],
        "archivo_disponible": archivo.exists(),
        "geometria_valida": False,
        "detalle": ""
    }

    if not archivo.exists():
        resultado["detalle"] = "Archivo pendiente"
        revision.append(resultado)
        continue

    try:
        geometria_json = json.loads(
            archivo.read_text(encoding="utf-8")
        )

        if geometria_json.get("type") != "Polygon":
            raise ValueError("Se esperaba una geometría Polygon.")

        geometria = shape(geometria_json)

        valida = (
            geometria.is_valid
            and not geometria.is_empty
        )

        resultado["geometria_valida"] = valida
        resultado["detalle"] = explain_validity(geometria)

        if valida:
            geometrias[fila["parcela_id"]] = geometria

    except Exception as error:
        resultado["detalle"] = str(error)

    revision.append(resultado)

revision_df = pd.DataFrame(revision)

display(revision_df)

if not revision_df["archivo_disponible"].all():
    raise FileNotFoundError(
        "Faltan archivos de geometría. Revisa la tabla."
    )

if not revision_df["geometria_valida"].all():
    raise ValueError(
        "Hay geometrías no válidas. Revisa la columna detalle."
    )

catalogo["estado_revision"] = (
    catalogo["parcela_id"]
    .map(
        revision_df.set_index("parcela_id")["geometria_valida"]
    )
    .map({
        True: "Geometría validada",
        False: "Revisar"
    })
)

print("Las cuatro geometrías están disponibles y son válidas.")

## 11. Detección de geometrías coincidentes

Compara los polígonos cargados. Una tabla vacía indica que no hay parejas geométricamente iguales entre las entradas del catálogo. No elimina registros automáticamente.


In [ ]:
coincidencias = []
ids = list(geometrias)

for i, parcela_a in enumerate(ids):
    for parcela_b in ids[i + 1:]:
        if geometrias[parcela_a].equals(geometrias[parcela_b]):
            coincidencias.append({
                "parcela_a": parcela_a,
                "parcela_b": parcela_b,
                "incidencia": "Misma geometría"
            })

coincidencias_df = pd.DataFrame(
    coincidencias,
    columns=["parcela_a", "parcela_b", "incidencia"]
)

display(coincidencias_df)
print("Parcelas del catálogo:", len(catalogo))
print("Geometrías válidas cargadas:", len(geometrias))
print("Pares con geometría coincidente:", len(coincidencias_df))

## 12. Selección por nombre o identificador

Normaliza nombres para admitir mayúsculas, acentos, guiones y guiones bajos. seleccionar_parcela devuelve los metadatos y la geometría leída desde Drive. La selección debe corresponder a una única entrada.


In [ ]:
import unicodedata


def normalizar_nombre(texto):
    texto = unicodedata.normalize("NFKD", str(texto))

    texto = "".join(
        caracter
        for caracter in texto
        if not unicodedata.combining(caracter)
    )

    texto = (
        texto.strip()
        .casefold()
        .replace("_", " ")
        .replace("-", " ")
    )

    return " ".join(texto.split())


def seleccionar_parcela(nombre_o_id):
    busqueda = normalizar_nombre(nombre_o_id)

    mascara = (
        catalogo["nombre"].map(normalizar_nombre).eq(busqueda)
        | catalogo["parcela_id"].map(normalizar_nombre).eq(busqueda)
    )

    coincidencias = catalogo.loc[mascara]

    if len(coincidencias) != 1:
        disponibles = ", ".join(
            catalogo["nombre"].tolist()
        )

        raise ValueError(
            f"No se reconoce una parcela única para '{nombre_o_id}'. "
            f"Parcelas disponibles: {disponibles}"
        )

    parcela = coincidencias.iloc[0].to_dict()
    archivo = parcelas_path / parcela["archivo_geometria"]

    if not archivo.exists():
        raise FileNotFoundError(
            f"No existe el archivo {archivo.name}."
        )

    geometria_json = json.loads(
        archivo.read_text(encoding="utf-8")
    )

    if geometria_json.get("type") != "Polygon":
        raise ValueError("Se esperaba una geometría Polygon.")

    geometria = shape(geometria_json)

    if geometria.is_empty or not geometria.is_valid:
        raise ValueError(
            f"Geometría no válida: {explain_validity(geometria)}"
        )

    parcela["geometria"] = geometria_json

    print("Parcela:", parcela["nombre"])
    print("Identificador:", parcela["parcela_id"])
    print("Municipio:", parcela["municipio"])
    print("Variedad:", parcela["variedad"])
    print("Archivo:", archivo.name)
    print("Geometría válida:", geometria.is_valid)

    return parcela

## 13. Prueba de selección de Pancostales

Verifica el identificador, el cierre del anillo y una coordenada de referencia. Los assert detienen la prueba si no se cumple lo esperado. No sustituye a revisar todas las coordenadas ni a contrastarlas en campo.


In [ ]:
parcela_pancostales = seleccionar_parcela("Pancostales")

anillo = parcela_pancostales["geometria"]["coordinates"][0]

assert parcela_pancostales["parcela_id"] == "pancostales"
assert anillo[0] == anillo[-1]
assert anillo[0] == json.loads(GEOMETRIAS["pancostales"])["coordinates"][0][0]

print("\nPrueba de Pancostales superada.")

## 14. Prueba de Los Ceres y unicidad del catálogo

Verifica la selección de Los Ceres, una coordenada de referencia, cuatro registros y ausencia de identificadores repetidos.


In [ ]:
parcela_ceres = seleccionar_parcela("Los Ceres")

anillo = parcela_ceres["geometria"]["coordinates"][0]

assert parcela_ceres["parcela_id"] == "los_ceres"
assert anillo[0] == anillo[-1]
assert anillo[0] == json.loads(GEOMETRIAS["los_ceres"])["coordinates"][0][0]

assert len(catalogo) == 4
assert catalogo["parcela_id"].is_unique

print("\nPrueba de Los Ceres superada.")
print("Catálogo único de cuatro parcelas confirmado.")

## 15. Exportación del catálogo y de las revisiones

Guarda catalogo_parcelas.csv y los informes de geometrías y coincidencias. Ejecutar después de las comprobaciones. El notebook de extracción leerá este catálogo y sus GeoJSON; no necesita copiar las coordenadas otra vez.


In [ ]:
catalogo.to_csv(
    parcelas_path / "catalogo_parcelas.csv",
    index=False,
    encoding="utf-8-sig"
)

revision_df.to_csv(
    resultados_path / "revision_geometrias.csv",
    index=False,
    encoding="utf-8-sig"
)

coincidencias_df.to_csv(
    resultados_path / "coincidencias_geometrias.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivos guardados:")
print("- catalogo_parcelas.csv")
print("- revision_geometrias.csv")
print("- coincidencias_geometrias.csv")

## 16. Registro privado del avance

Escribe una nota de trabajo con el alcance, la fusión de duplicados y lo pendiente. Ejecutar solo tras completar correctamente las comprobaciones. La fecha y las afirmaciones del texto corresponden al trabajo documentado; si se reutiliza para otra sesión deben revisarse. Esta operación reemplaza el archivo registro_avances.md.


In [ ]:
registro = """# Preparación de parcelas

## Catálogo operativo

Se ha preparado un catálogo de cuatro parcelas:

- Pancostales.
- Puri.
- La Culebra.
- Los Ceres.

## Comprobaciones

Se han comprobado la disponibilidad y la validez de las geometrías.
Se han ejecutado pruebas de selección para Pancostales y Los Ceres.

Los resultados de la revisión se conservan en:
- revision_geometrias.csv
- coincidencias_geometrias.csv

## Alcance

Los históricos de sensores y riego quedan fuera de esta primera etapa.

Este notebook prepara y valida el catálogo. No extrae todavía imágenes
Sentinel-2 ni genera recomendaciones de riego.

## Pendiente

- Confirmar la campaña o intervalo de extracción.
- Adaptar Sentinel-2 al catálogo de cuatro parcelas.
- Revisar la escala de reflectancia para calcular EVI y SAVI.
- Revisar las entradas y salidas del motor original.
"""

(base_path / "registro_avances.md").write_text(
    registro,
    encoding="utf-8"
)

print("Registro actualizado.")

## 17. Selección manual de comprobación

Ejemplo de lectura de una parcela preparada. NOMBRE_PARCELA solo afecta a esta comprobación manual. El extractor automático posterior recorre el catálogo completo y no depende de esta selección.


In [ ]:
NOMBRE_PARCELA = "Los Ceres"

parcela_actual = seleccionar_parcela(NOMBRE_PARCELA)

print(
    "\nParcela preparada para el futuro extractor:",
    parcela_actual["nombre"]
)

## Cierre y siguiente cuaderno
Tras superar los controles, conservar los cuatro GeoJSON y catalogo_parcelas.csv en la carpeta parcelas de Drive. El notebook 03 realiza la extracción automática leyendo esos archivos; no es necesario volver a pegar las coordenadas allí. Antes de publicar este notebook en el repositorio, borrar sus salidas y comprobar que no contiene coordenadas, municipios ni variedades.
